# JustIA – Actividad 1
## Clasificación automática de casos jurídicos con BETO (BERT en español)
**Corporación Universitaria de Asturias**

**Objetivo:** Aplicar el modelo preentrenado `dccuchile/bert-base-spanish-wwm-cased` para clasificar fragmentos de sentencias jurídicas por área temática: **penal, civil, laboral, familia**.

---

## 0. Instalación de dependencias

In [1]:
# Ejecutar solo la primera vez
!pip install transformers datasets scikit-learn torch accelerate --quiet


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Importaciones

In [10]:
import random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("✅ Librerías cargadas correctamente")
print(f"🖥️  Dispositivo: {'GPU' if torch.cuda.is_available() else 'CPU'}")

✅ Librerías cargadas correctamente
🖥️  Dispositivo: GPU


## 2. Generación del dataset simulado (200 fragmentos jurídicos)

Se simulan fragmentos representativos de sentencias colombianas en cuatro áreas del derecho.

In [3]:
# ─── Plantillas de texto por categoría ───────────────────────────────────────

textos_penal = [
    "El procesado fue capturado en flagrancia portando un arma de fuego sin permiso de porte.",
    "La Fiscalía General de la Nación formuló cargos por el delito de homicidio agravado.",
    "Se decretó la medida de aseguramiento de detención preventiva en establecimiento carcelario.",
    "El imputado aceptó cargos por el delito de hurto calificado y agravado.",
    "La sentencia condenó al acusado a doce años de prisión por el delito de tráfico de estupefacientes.",
    "El Juez Penal del Circuito declaró la responsabilidad penal del enjuiciado por lesiones personales.",
    "Se emitió orden de captura contra el indiciado por el delito de extorsión continuada.",
    "El testimonio de la víctima fue determinante para establecer la culpabilidad del acusado.",
    "Se absolvió al procesado por atipicidad de la conducta descrita en la acusación.",
    "La defensa solicitó la nulidad procesal alegando violación al debido proceso penal.",
    "El delito de feminicidio fue tipificado conforme al artículo 104A del Código Penal colombiano.",
    "El juez ordenó la preclusión de la investigación por ausencia de mérito probatorio.",
    "Se impuso pena accesoria de inhabilitación para el ejercicio de derechos y funciones públicas.",
    "El procesado interpuso recurso de apelación contra la sentencia condenatoria de primera instancia.",
    "La Sala Penal del Tribunal Superior confirmó la condena impuesta en primera instancia.",
    "Se decretó la extinción de dominio sobre bienes de origen ilícito provenientes del narcotráfico.",
    "El sistema de responsabilidad penal para adolescentes impuso medida pedagógica al infractor.",
    "Se reconoció circunstancia de menor punibilidad por estado de ira e intenso dolor.",
    "La conducta de peculado por apropiación fue atribuida al servidor público en ejercicio del cargo.",
    "El acusado fue condenado como coautor del delito de concierto para delinquir agravado.",
    "Se solicitó preclusión por muerte del procesado durante el trámite del proceso penal.",
    "La Corte Suprema de Justicia Sala Penal casó parcialmente la sentencia del Tribunal.",
    "El ente acusador probó el dolo directo en la comisión del delito de secuestro extorsivo.",
    "Se aplicó el principio de favorabilidad ante la vigencia de una norma más benéfica.",
    "El juez de ejecución de penas y medidas de seguridad concedió la libertad condicional.",
    "El delito de violencia intrafamiliar fue calificado como conducta de mayor gravedad.",
    "La Fiscalía formuló acusación por el delito de abuso de confianza agravado.",
    "Se negó la sustitución de la pena privativa de libertad por prisión domiciliaria.",
    "El procesado fue condenado por el concurso de delitos de falsedad y estafa.",
    "El juez promiscuo municipal declaró la prescripción de la acción penal.",
    "Se interpuso tutela contra providencia judicial que vulneró el debido proceso.",
    "La conducta constituyó tentativa de homicidio en grado de tentativa acabada.",
    "Se reconoció la condición de víctima del conflicto armado para efectos reparatorios.",
    "El imputado suscribió preacuerdo con la Fiscalía por el delito de receptación.",
    "El juez de control de garantías avaló la legalidad de la captura del indiciado.",
    "Se dispuso la destrucción de la droga incautada conforme al protocolo legal.",
    "La pena de multa fue tasada conforme a los criterios del Código Penal vigente.",
    "Se declaró la inimputabilidad del procesado por trastorno mental permanente.",
    "El juez ordenó la restitución del bien inmueble despojado a la víctima del conflicto.",
    "Se aplicó la rebaja por confesión anticipada conforme al artículo 351 de la Ley 906.",
    "La Sala admitió el recurso extraordinario de casación por violación directa de la ley.",
    "El peritaje forense estableció la data de la lesión como anterior al hecho denunciado.",
    "Se impuso la inhabilitación para el ejercicio de profesiones liberales por diez años.",
    "La conducta de lavado de activos fue probada mediante análisis de movimientos bancarios.",
    "El juez rechazó la nulidad por saneamiento tácito de la irregularidad alegada.",
    "Se negó la exclusión de evidencia ilícita al verificarse la excepción de buena fe.",
    "El Tribunal ordenó la compulsa de copias por posible responsabilidad penal del testigo.",
    "La víctima fue reconocida como interviniente especial en el proceso penal acusatorio.",
    "El juez decretó comiso de los instrumentos utilizados en la comisión del delito.",
    "Se acreditó la calidad de servidor público como elemento normativo del tipo penal.",
]

textos_civil = [
    "Se declaró la resolución del contrato de compraventa por incumplimiento del vendedor.",
    "El demandante solicitó la indemnización de perjuicios por responsabilidad civil extracontractual.",
    "El juez civil del circuito decretó el embargo y secuestro de bienes del deudor.",
    "Se admitió la demanda ejecutiva con título hipotecario para el cobro de obligación dineraria.",
    "La sucesión intestada fue tramitada conforme a las disposiciones del Código Civil.",
    "Se declaró la nulidad absoluta del contrato por objeto ilícito conforme al artículo 1519.",
    "El demandado opuso la excepción de prescripción extintiva de la acción civil.",
    "Se ordenó la restitución del inmueble arrendado por mora en el pago del canon.",
    "El juez accedió a la acción de tutela patrimonial por riesgo de lesión a derechos reales.",
    "Se reconoció la servidumbre de tránsito sobre el predio sirviente del demandado.",
    "La Sala Civil del Tribunal Superior negó las pretensiones por falta de legitimación en causa.",
    "Se profirió sentencia estimatoria en acción de simulación de contrato de donación.",
    "El perito avaluador estimó los perjuicios materiales en la suma de ciento veinte millones.",
    "Se declaró la prescripción adquisitiva de dominio por posesión quieta y pacífica de veinte años.",
    "El acreedor hipotecario solicitó el avalúo y remate del bien dado en garantía.",
    "Se reconoció la solidaridad pasiva entre los codeudores de la obligación incumplida.",
    "La acción pauliana prosperó al acreditarse el fraude en la enajenación del bien.",
    "El juez de familia conoció del proceso de sucesión por fuero de atracción.",
    "Se tasaron los perjuicios morales en el equivalente a cien salarios mínimos mensuales.",
    "La demanda de pertenencia fue inadmitida por no acreditar el tiempo de posesión requerido.",
    "Se ordenó la cancelación del registro de la escritura pública de compraventa.",
    "El demandante acreditó el daño emergente y el lucro cesante con dictamen pericial.",
    "Se negó la medida cautelar de embargo por no haberse prestado la caución requerida.",
    "La acción de responsabilidad civil médica fue acogida parcialmente por culpa probada.",
    "Se decretó la nulidad del testamento por incapacidad del testador al momento del otorgamiento.",
    "El contrato de arrendamiento fue declarado terminado por mutuo disenso de las partes.",
    "Se reconoció la obligación natural convertida en civil por novación expresa.",
    "El juez ordenó la rendición de cuentas por parte del administrador de la sociedad.",
    "La responsabilidad del Estado fue declarada por daño especial en obra pública.",
    "Se profirió auto admisorio del proceso de insolvencia de persona natural no comerciante.",
    "El deudor fue declarado en cesación de pagos y se abrió el proceso de liquidación judicial.",
    "Se reconoció la ineficacia de la cláusula de limitación de responsabilidad por dolo.",
    "La condena en costas fue impuesta al demandado vencido en ambas instancias.",
    "Se declaró probada la excepción de contrato no cumplido opuesta por el demandado.",
    "El juez ordenó el levantamiento de las medidas cautelares al cancelarse la deuda.",
    "Se reconoció la mejora de buena fe realizada por el poseedor sobre el inmueble reivindicado.",
    "La acción de responsabilidad por producto defectuoso fue declarada procedente.",
    "Se admitió la reconvención por incumplimiento contractual del demandante original.",
    "El remate del bien inmueble fue suspendido por solicitud de amparo de pobreza.",
    "Se declaró la simulación relativa del contrato de compraventa encubriendo una donación.",
    "La Corte Suprema casa la sentencia por error de hecho en la apreciación de la prueba.",
    "Se reconoció la cláusula penal como estimación anticipada de perjuicios.",
    "El notario se abstuvo de autorizar la escritura por objeto contrario a la ley.",
    "Se accedió a la interdicción judicial del demente por incapacidad mental absoluta.",
    "La acción reivindicatoria prosperó al acreditarse la identidad del bien y la posesión ajena.",
    "Se declaró la nulidad relativa del contrato por error esencial sobre la calidad del objeto.",
    "El acreedor prendario ejerció la acción de cobro ejecutivo sobre el bien mueble dado en prenda.",
    "Se reconoció la comunidad de bienes sobre el inmueble adquirido durante la unión marital.",
    "El juez aprobó el acuerdo de pago suscrito entre acreedor y deudor en proceso ejecutivo.",
    "Se ordenó la acumulación de procesos ejecutivos seguidos contra el mismo deudor.",
]

textos_laboral = [
    "El trabajador fue despedido sin justa causa, por lo que se ordenó el pago de indemnización.",
    "Se reconoció la relación laboral encubierta bajo la figura de contrato de prestación de servicios.",
    "El empleador fue condenado al pago de salarios, prestaciones sociales y aportes a seguridad social.",
    "Se declaró la nulidad del despido colectivo por omisión del trámite ante el Ministerio del Trabajo.",
    "El trabajador acreditó el acoso laboral mediante pruebas testimoniales y documentales.",
    "Se ordenó el reintegro del trabajador amparado por fuero de salud al cargo que desempeñaba.",
    "La empresa fue sancionada por no afiliar al trabajador al sistema de riesgos laborales.",
    "Se liquidó la cesantía conforme al último salario devengado por el trabajador.",
    "El juez laboral ordenó el pago de las horas extras laboradas durante tres años.",
    "Se reconoció la pensión de invalidez al trabajador con pérdida de capacidad laboral superior al 50%.",
    "La huelga fue declarada ilegal por no cumplir los requisitos del Código Sustantivo del Trabajo.",
    "Se ordenó el pago de la prima de servicios proporcional al tiempo trabajado.",
    "El trabajador obtuvo el reconocimiento de la pensión de vejez por haber cumplido los requisitos legales.",
    "Se declaró la existencia del contrato de trabajo por la presunción del artículo 24 del CST.",
    "La Sala Laboral del Tribunal condenó al empleador por mora en el pago de acreencias laborales.",
    "Se reconoció el fuero sindical al trabajador que ejercía representación gremial.",
    "El empleador no acreditó justa causa para dar por terminado el contrato de trabajo.",
    "Se ordenó el pago de la indemnización moratoria por retención injustificada de salarios.",
    "El trabajador fue discriminado por razón de su estado de salud en el proceso de selección.",
    "Se declaró la sustitución de empleadores y la continuidad de la relación laboral.",
    "El juez laboral de circuito rechazó la excepción de prescripción de tres años propuesta.",
    "Se reconoció el salario variable para efectos de la liquidación de prestaciones sociales.",
    "La empresa fue condenada al pago de dotación no entregada durante la relación laboral.",
    "Se ordenó el reintegro del trabajador amparado por fuero de maternidad de la cónyuge.",
    "El contrato a término fijo fue prorrogado tácitamente por continuar la prestación del servicio.",
    "Se reconoció la responsabilidad del empleador por accidente de trabajo con culpa patronal.",
    "La Corte Suprema Sala Laboral unificó jurisprudencia sobre el contrato realidad.",
    "Se declaró la ineficacia del período de prueba por exceder el término legal permitido.",
    "El empleador fue condenado por no consignar oportunamente las cesantías al fondo.",
    "Se reconoció el auxilio de conectividad al teletrabajador conforme a la ley 2121 de 2021.",
    "El trabajador solicitó la reliquidación de la pensión incluyendo factores salariales omitidos.",
    "Se ordenó la indexación de las sumas adeudadas desde la terminación del contrato.",
    "La empresa respondió solidariamente por las obligaciones del contratista respecto del trabajador.",
    "Se declaró la existencia de un grupo empresarial para efectos de la solidaridad laboral.",
    "El juez reconoció la jornada máxima legal y el recargo nocturno al trabajador de turno.",
    "Se accedió parcialmente a las pretensiones por prescripción de algunas acreencias.",
    "La acción de tutela prosperó para proteger el mínimo vital del trabajador despedido.",
    "El inspector de trabajo levantó acta de infracción por violación a normas de seguridad.",
    "Se reconoció la pensión de sobrevivientes a la compañera permanente del trabajador fallecido.",
    "El empleador fue absuelto por demostrar la configuración de justa causa de despido.",
    "Se ordenó la reliquidación de las prestaciones incluyendo las comisiones de ventas.",
    "La Sala declaró la solidaridad de la empresa principal frente al contratista independiente.",
    "Se reconoció el derecho de asociación sindical como garantía constitucional del trabajador.",
    "El juez ordenó el reintegro por nulidad del despido durante período de incapacidad médica.",
    "Se confirmó la legalidad del reglamento interno de trabajo adoptado por la empresa.",
    "La pensión convencional fue reconocida conforme a la convención colectiva vigente.",
    "Se tasaron los perjuicios por enfermedad laboral causada por exposición a ruido industrial.",
    "El trabajador acreditó haber laborado horas dominicales sin el recargo legal correspondiente.",
    "Se reconoció la reliquidación de la mesada pensional conforme al IPC actualizado.",
    "El juez de pequeñas causas laborales falló a favor del trabajador en audiencia única.",
]

textos_familia = [
    "Se decretó el divorcio por mutuo acuerdo y se aprobó la liquidación de la sociedad conyugal.",
    "El juez de familia reconoció la unión marital de hecho por convivencia superior a dos años.",
    "Se fijó cuota alimentaria provisional a cargo del padre en favor del menor de edad.",
    "La demanda de custodia y cuidado personal fue resuelta otorgando custodia compartida.",
    "Se decretó la medida de restablecimiento de derechos del niño en situación de abandono.",
    "El ICBF intervino como autoridad administrativa para proteger los derechos del adolescente.",
    "Se reconoció la paternidad extramatrimonial mediante prueba de ADN con valor probatorio del 99.9%.",
    "El juez de familia ordenó la disolución y liquidación de la sociedad patrimonial entre compañeros.",
    "Se declaró la interdicción judicial de persona con discapacidad cognitiva severa.",
    "El proceso de adopción fue aprobado tras verificarse el interés superior del menor.",
    "Se decretó la separación de bienes por dilapidación del patrimonio familiar por parte del cónyuge.",
    "La medida de protección por violencia intrafamiliar fue ratificada en audiencia de seguimiento.",
    "Se ordenó la restitución internacional del menor retenido ilícitamente en el exterior.",
    "El juez promiscuo de familia declaró desierta la demanda por inasistencia a la audiencia.",
    "Se reconoció la vocación hereditaria del hijo extramatrimonial reconocido voluntariamente.",
    "La curadoría fue asignada al familiar más cercano capaz de asumir la responsabilidad.",
    "Se aprobó el acuerdo de alimentos suscrito entre los progenitores del menor beneficiario.",
    "El régimen de visitas fue modificado atendiendo el interés superior del niño afectado.",
    "Se declaró la nulidad del matrimonio civil por error en la identidad del contrayente.",
    "El proceso de insolvencia familiar fue resuelto mediante acuerdo de pago supervisado.",
    "Se reconoció el daño moral sufrido por los hijos menores por la separación abrupta del padre.",
    "La demanda de impugnación de paternidad fue rechazada por extemporánea.",
    "Se ordenó la suspensión de la patria potestad por maltrato reiterado acreditado en el proceso.",
    "El juez de familia ordenó el registro civil de nacimiento del menor no inscrito.",
    "Se declaró la cesación de los efectos civiles del matrimonio religioso entre las partes.",
    "La cuota alimentaria fue indexada conforme al incremento del IPC del año anterior.",
    "Se reconoció el estado civil de hijo adoptivo con todos los derechos sucesorales.",
    "El comisario de familia ordenó medida de protección por violencia económica en el hogar.",
    "Se ratificó la custodia materna al verificarse el apego y bienestar del menor con la progenitora.",
    "El juez aprobó la capitulación matrimonial pactada antes de la celebración del matrimonio.",
    "Se declaró la extinción de la obligación alimentaria por mayoría de edad del beneficiario.",
    "La audiencia de conciliación familiar fracasó y el proceso continuó en sede judicial.",
    "Se reconoció el derecho de visitas del abuelo como parte de las relaciones parentales amplias.",
    "El juez ordenó la práctica de valoración psicológica al menor para determinar la custodia.",
    "Se declaró la nulidad de la capitulación por vicios del consentimiento en su otorgamiento.",
    "El proceso de liquidación de la sociedad conyugal fue aprobado en audiencia de inventarios.",
    "Se reconoció la porción conyugal al cónyuge supérstite pobre en el proceso sucesoral.",
    "La defensoría de familia emitió concepto sobre condiciones de vulnerabilidad del menor.",
    "Se accedió a la solicitud de cambio de nombre por homonimia demostrada documentalmente.",
    "El juez rechazó la demanda de divorcio por no haberse demostrado la causal alegada.",
    "Se ordenó el pago de alimentos provisionales durante el trámite del proceso de separación.",
    "La medida de restablecimiento fue levantada al verificarse la superación del riesgo familiar.",
    "Se aprobó el acuerdo de custodia presentado por los progenitores ante el juez de familia.",
    "El curador ad litem fue designado para representar los intereses del menor en el proceso.",
    "Se reconoció la sociedad patrimonial disuelta por separación de hecho superior a dos años.",
    "El proceso voluntario de permiso de salida del país del menor fue aprobado por el juez.",
    "Se negó el divorcio unilateral por no cumplirse el término de separación legal exigido.",
    "La Sala de Familia del Tribunal confirmó la custodia otorgada al padre como cuidador principal.",
    "Se impuso medida de internamiento terapéutico al adolescente con consumo problemático.",
    "El juez de familia homologó la sentencia extranjera de divorcio conforme al exequátur.",
]

# Construir DataFrame con 200 registros (50 por clase)
categorias = {
    "penal":   textos_penal[:50],
    "civil":   textos_civil[:50],
    "laboral": textos_laboral[:50],
    "familia": textos_familia[:50],
}

registros = []
for etiqueta, textos in categorias.items():
    for t in textos:
        registros.append({"texto": t, "etiqueta": etiqueta})

df = pd.DataFrame(registros).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Total registros: {len(df)}")
print(df["etiqueta"].value_counts())
df.head(6)

Total registros: 200
etiqueta
civil      50
penal      50
familia    50
laboral    50
Name: count, dtype: int64


,texto,etiqueta
0,Se declaró la nulidad relativa del contrato po...,civil
1,Se decretó la extinción de dominio sobre biene...,penal
2,Se interpuso tutela contra providencia judicia...,penal
3,Se declaró la interdicción judicial de persona...,familia
4,El empleador fue condenado por no consignar op...,laboral
5,Se reconoció el fuero sindical al trabajador q...,laboral


## 3. Preprocesamiento y codificación de etiquetas

In [4]:
# Mapeo de etiquetas a enteros
LABEL2ID = {"civil": 0, "familia": 1, "laboral": 2, "penal": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df["label"] = df["etiqueta"].map(LABEL2ID)

# División train / test (80/20)
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=SEED
)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")

Train: 160 | Test: 40


## 4. Carga del tokenizador BETO

In [5]:
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["texto"],
        truncation=True,
        padding=False,       # el DataCollator se encarga del padding dinámico
        max_length=128,
    )

# Convertir a Dataset de HuggingFace
train_ds = Dataset.from_pandas(train_df[["texto", "label"]])
test_ds  = Dataset.from_pandas(test_df[["texto", "label"]])

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print("✅ Tokenización completada")
print(train_ds)

c:\Users\milli\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\milli\.cache\huggingface\hub\models--dccuchile--bert-base-spanish-wwm-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 40/40 [00:00<00:00, 2399.11 examples/s]

✅ Tokenización completada
Dataset({
    features: ['texto', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 160
})


## 5. Carga del modelo BETO para clasificación de secuencias

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print(f"✅ Modelo cargado: {MODEL_NAME}")
print(f"   Parámetros totales: {model.num_parameters():,}")

ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

## 6. Fine-tuning con Hugging Face Trainer

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}


training_args = TrainingArguments(
    output_dir="./justia_beto_output",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_dir="./logs",
    logging_steps=20,
    seed=SEED,
    report_to="none",     # cambiar a 'wandb' si se desea monitoreo externo
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🚀 Iniciando fine-tuning de BETO...")
trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


NameError: name 'model' is not defined

## 7. Evaluación final en conjunto de prueba

In [ ]:
# Métricas globales
resultados = trainer.evaluate()
print("=" * 50)
print(f"  Accuracy : {resultados['eval_accuracy']:.4f}")
print(f"  F1 macro : {resultados['eval_f1_macro']:.4f}")
print("=" * 50)

In [ ]:
# Reporte detallado por clase
predicciones = trainer.predict(test_ds)
preds_ids    = np.argmax(predicciones.predictions, axis=-1)
true_ids     = predicciones.label_ids

etiquetas_nombres = [ID2LABEL[i] for i in range(4)]

print("\n📊 Reporte de clasificación por clase:")
print(classification_report(true_ids, preds_ids, target_names=etiquetas_nombres))

## 8. Visualización de la matriz de confusión

In [ ]:
cm = confusion_matrix(true_ids, preds_ids)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=etiquetas_nombres,
    yticklabels=etiquetas_nombres,
    ax=ax,
)
ax.set_title("Matriz de Confusión – JustIA BETO", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicción")
ax.set_ylabel("Etiqueta real")
plt.tight_layout()
plt.savefig("matriz_confusion_justia.png", dpi=150)
plt.show()
print("Imagen guardada: matriz_confusion_justia.png")

## 9. Predicción sobre ejemplos nuevos

In [ ]:
ejemplos_nuevos = [
    "La víctima del conflicto solicitó reparación por los crímenes cometidos por grupos armados.",
    "Se ordenó el pago de los salarios adeudados al trabajador durante el período de incapacidad.",
    "El arrendatario demandó la devolución del depósito por terminación normal del contrato.",
    "Los menores fueron declarados en situación de abandono y se inició proceso de adoptabilidad.",
    "El procesado fue condenado por el delito de tráfico de estupefacientes en zona escolar.",
]

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("🔍 Predicciones sobre textos nuevos:")
print("-" * 80)
for texto in ejemplos_nuevos:
    inputs = tokenizer(
        texto, return_tensors="pt", truncation=True,
        padding=True, max_length=128
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    prob      = torch.softmax(logits, dim=-1)[0]
    pred_id   = torch.argmax(prob).item()
    pred_cat  = ID2LABEL[pred_id]
    confianza = prob[pred_id].item()
    print(f"Texto    : {texto[:70]}...")
    print(f"Categoría: {pred_cat.upper():10s}  |  Confianza: {confianza:.2%}")
    print("-" * 80)

## 10. Resumen de métricas y reflexión crítica

### Resultados obtenidos

| Métrica | Valor |
|---------|-------|
| Accuracy | Ver celda 7 |
| F1-Score Macro | Ver celda 7 |

### Reflexión crítica para JustIA

**Fortalezas del enfoque:**
- BETO es un modelo preentrenado sobre corpus en español (incluye textos legales de prensa y Wikipedia), lo que reduce el costo de entrenamiento.
- El fine-tuning con datos jurídicos colombianos permite que el modelo aprenda la terminología específica del ordenamiento legal nacional.
- La tokenización a nivel de subpalabras (WordPiece) maneja bien neologismos jurídicos y variantes morfológicas del español.

**Limitaciones y riesgos éticos:**
1. **Sesgo en datos:** Si los fragmentos de entrenamiento provienen solo de ciertos circuitos judiciales o períodos históricos, el modelo podría no generalizar a otras regiones o contextos.
2. **Términos ambiguos:** El modelo puede confundir áreas si un mismo concepto aparece en varias (ej. "violencia" en penal y familia).
3. **Riesgo de sobreinterpretación:** Una clasificación automática no reemplaza el análisis jurídico del abogado; debe usarse como filtro de apoyo, no como decisión definitiva.
4. **Poblaciones vulnerables:** Dado que JustIA atiende a víctimas del conflicto y comunidades indígenas, los errores de clasificación pueden derivar en orientación jurídica incorrecta con consecuencias reales.

**Recomendaciones para JustIA:**
- Implementar umbral de confianza mínimo (~0.80); si el modelo no supera ese umbral, derivar el caso a un practicante humano.
- Mantener trazabilidad de cada predicción con la probabilidad asociada y la versión del modelo utilizado.
- Realizar auditorías periódicas comparando predicciones con la clasificación final del abogado.
- Ampliar el dataset con sentencias del Consejo de Estado y la Corte Constitucional colombiana para mejorar la cobertura temática.

---
*Proyecto JustIA – Corporación Universitaria de Asturias | Caso Práctico Unidad 2*